# Generate Summaries with Fine-tuned Mistral-7B

Loads the QLoRA adapter and generates summaries for 200 products.
Download the results and run LLM-as-Judge + A/B test locally.

**Upload these files:**
1. `summary_training_pairs.jsonl` (from data/processed/)
2. `mistral-qlora-adapter.zip` (from models/)

In [ ]:
# Step 1: Upload files
from google.colab import files
uploaded = files.upload()  # Upload both files

In [ ]:
# Step 2: Install deps and unzip adapter
!pip install -q transformers peft bitsandbytes accelerate
!unzip -q mistral-qlora-adapter.zip

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Load base model in 4-bit
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print("Base model loaded")

In [ ]:
# Step 4: Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, "./mistral-qlora-adapter")
model.eval()
print("Adapter loaded")

In [ ]:
# Step 5: Load training pairs (we generate for same products)
pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))
print(f"Loaded {len(pairs)} products")

In [ ]:
# Step 6: Generate summaries
def generate_summary(model, tokenizer, reviews_input):
    prompt = (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{reviews_input[:3000]}\n[/INST]\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated.split('[/INST]')[-1].strip()
    return response

results = []
for i, pair in enumerate(pairs):
    summary = generate_summary(model, tokenizer, pair['input'])
    results.append({
        'asin': pair['asin'],
        'product_title': pair.get('product_title', ''),
        'input': pair['input'],
        'summary': summary,
    })
    if (i + 1) % 10 == 0:
        print(f"Generated {i + 1} / {len(pairs)}")
        # Save progress
        with open('finetuned_summaries.jsonl', 'w') as f:
            for r in results:
                f.write(json.dumps(r) + '\n')

# Final save
with open('finetuned_summaries.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f"Done! {len(results)} summaries generated")

In [ ]:
# Step 7: Quick sample check
print("=== Sample Output ===")
print(f"Product: {results[0]['product_title']}")
print(f"Summary:\n{results[0]['summary'][:500]}")

In [ ]:
# Step 8: Download results
files.download('finetuned_summaries.jsonl')
print("\nSave to data/processed/finetuned_summaries.jsonl in your project")